## 📑 Core DPO Campaign
 
#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/x/AYAOH)**  
  Comprehensive documentation outlining the business requirements for this report.


##### Authentication

In [0]:
%python
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
%python
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col,regexp_replace

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("example").getOrCreate()

import pandas as pd
from pyspark.sql import DataFrame
from pyspark.sql.functions import trim

In [0]:
%python
import boto3
import os

#takes the values d for dev, qa for test and p for prod depending in which workspace the init variables script is running
# env = os.environ["S3PREFIX"]

dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure 
                         union
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.landed.profits_loan_details
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d') 

###### Schema of the table

In [0]:
%python
table_name = 'Core_DPOCampaign'

schema = (
"BusinessDate date,"
"ExposureKey string,"
"NewExposureKey string,"
"BalanceGroup string,"
"SegmentDPO string,"
"EAPClass string,"
"AccountStatus string,"
"Active string,"
"Segment string,"
"Balance decimal(19,2),"
"InteractionTypeKey string,"
"ActivityCode string,"
"InteractionDate date,"
"Category string,"
"Subcategory string,"
"RestComms string,"
"Modification string,"
"ModificationDate date,"
"ModsProduct string,"
"ModsHaircut decimal(19,2),"
"ModsAmount decimal(19,2),"
"ModsAmountBeforeModification decimal(19,2),"
"Status string,"
"DPOProduct string,"
"ModificationType string,"
"ApplicationNumber string,"
"Decision string,"
"CurrentStepUpdated string,"
"ApplicationStep string,"
"DateEvaluation date,"
"ApplicationsFlag string,"
"ModificationFlag string,"
"OperationalBucket string,"
"RiskBucket string,"
"EBAStatus string,"
"DenouncedStatus string,"
"Subcategory2 string,"
"InsertTimeStamp timestamp"
)

In [0]:
%python
partition_columns = ["BusinessDate"]
delivery_table_path = ""
 
# ================== Create Delta Table ==================
delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"

delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema
)
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
  
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    run_date
#### type: ignore
)


##### Optimizer function

In [0]:
%python
# Helper function to create or replace table
def create_or_replace_table(table_name, df, partition_col):
    df.write.format("delta").mode("overwrite").partitionBy(partition_col).saveAsTable(table_name)
def create_or_replace_table_no_partition(table_name, df):
    df.write.format("delta").mode("overwrite").saveAsTable(table_name)
def repair_and_analyze_table(table_name):
    spark.sql(f"REPAIR TABLE {table_name} SYNC METADATA")
    spark.sql(f"ANALYZE TABLE {table_name} COMPUTE STATISTICS")    

##### Select Tables

In [0]:
%python
AssignmentPoolDPOCampaign = spark.sql(
f'''
SELECT 
     a.BusinessDate
    ,a.ExposureKey
    ,a.BalanceGroup
    ,a.SegmentDPO
    ,a.DenouncedFlag
    ,a.EAP
    ,a.Balance
    ,a.EBAStatus
    ,a.DPOModified
    ,a.ModificationMonth
    ,a.Segments
    ,a.ActiveCurrentMonthexclImplem
    from `{env_prefix}-cdh`.externalfiles.extassignmentpooldpocampaign a
    inner join (select max(IngestionTimeStamp) IngestionTimeStamp, BusinessDate from `{env_prefix}-cdh`.externalfiles.extassignmentpooldpocampaign group by BusinessDate) b
    on a.BusinessDate = b.BusinessDate and a.IngestionTimeStamp = b.IngestionTimeStamp
    where a.BusinessDate = (select max(BusinessDate) as BusinessDate from `{env_prefix}-cdh`.externalfiles.extassignmentpooldpocampaign where BusinessDate <= {run_date!r})
'''
)
AssignmentPoolDPOCampaign.createOrReplaceTempView("AssignmentPoolDPOCampaign")

# Check if the dataset is empty and print a message accordingly
if AssignmentPoolDPOCampaign.count() == 0:
    print(f"The required external file extassignmentpooldpocampaign dataset for dates <= {run_date!r} is empty")

casecode = spark.sql(
f'''
select distinct d.Portfolioname,  d.exposurekey ,d.casecode
from `{env_prefix}-cdh`.primary.casecode c
inner join `{env_prefix}-cdh`.primary.exposure d on c.casecodekey = d.casecode and c.PortfolioName = d.Portfolioname
where d.Portfolioname = 'Core Retail'
'''
)
casecode.createOrReplaceTempView("casecode")

interaction = spark.sql(
f'''
select
 l.ExposureKey
     , l.PortfolioName
     , l.InteractionDate
     , l.InteractionTypeKey
     , l.InteractionType
     , l.CDI
 FROM `{env_prefix}-cdh`.primary.get_interactions_by_business_date({run_date!r}) l
 inner join AssignmentPoolDPOCampaign a on l.ExposureKey = a.ExposureKey
where 1=1
and InteractionDate <= {run_date!r}
and InteractionDate >= '2022-03-28'
and l.SourceSystem='qca'
and l.PortfolioName = 'Core Retail'
and InteractionTypeKey IN ('140',	'152',	'201',	'215',	'216',	'3016',	'3017',	'3018',	'4014',	'4100',	'4101',	'4102',	'4103',	'4104',	'4105',	'4106',	'4107',	'4108',	'499',	'500',	'5002',	'501',	'514',	'516',	'518',	'524',	'525',	'529',	'531',	'533',	'534',	'536',	'537',	'545',	'546',	'547',	'571',	'572',	'573',	'577',	'578',	'777',	'780',	'781')
 
union
 
select
      c.ExposureKey
     , l.PortfolioName
     , l.InteractionDate
     , l.InteractionTypeKey
     , l.InteractionType
     , l.CDI
 FROM `{env_prefix}-cdh`.primary.get_interactions_by_business_date({run_date!r}) l
 inner join casecode c on c.casecode = l.casecode
 inner join AssignmentPoolDPOCampaign a on c.ExposureKey = a.ExposureKey
where 1=1
and InteractionDate <= {run_date!r}
and InteractionDate >= '2022-03-28'
and l.SourceSystem='qca'
and l.PortfolioName = 'Core Retail'
and InteractionTypeKey IN ('140',	'152',	'201',	'215',	'216',	'3016',	'3017',	'3018',	'4014',	'4100',	'4101',	'4102',	'4103',	'4104',	'4105',	'4106',	'4107',	'4108',	'499',	'500',	'5002',	'501',	'514',	'516',	'518',	'524',	'525',	'529',	'531',	'533',	'534',	'536',	'537',	'545',	'546',	'547',	'571',	'572',	'573',	'577',	'578',	'777',	'780',	'781')
''')
interaction.createOrReplaceTempView("interaction")

npeplanningdata = spark.sql(
f'''
select 
               AssetSegment
             , ExposureKey      
             , AgreementKey     
             , Decision			      
             , DateEvaluation       
             , MonthEvaluation    
             , ModificationType     
             , ApplicationNumber    
             , BalanceAmount        
             , AccountBalance
             , Status
             , ApplicationStep     
             , ApplicationStepFinal
             , ApplicationStepUpdated 
             , ReceiptDateCurrentStep    
 from `{env_prefix}-cdh-delivery`.repoint.core_npeplanningdata
where BusinessDate = {run_date!r}
'''
)
npeplanningdata.createOrReplaceTempView("npeplanningdata")

corestatichistoricalmodificationbeforemigration = spark.sql(
f'''
select 
    RestructuringDate 
    ,OldExposureKey
    ,NewExposureKey
    ,BusinessUnit
    ,RestructuringAmount
    ,HaircutAmount
    ,Category
    ,CategoryFinal
    ,Month
    ,IFRSGrossCarryingAmountPreviousMonthOldExposureKey
from `{env_prefix}-cdh`.reference.corestatichistoricalmodificationbeforemigration
'''
)
corestatichistoricalmodificationbeforemigration.createOrReplaceTempView("corestatichistoricalmodificationbeforemigration")

Core_Modifications = spark.sql(
f'''
select
          ModificationDate
        , BusinessDate
		, OldExposureKey 
		, NewExposureKey 
		, AssetSegment
		, ModificationAmount 
		, HaircutAmount 
		, Category 
		, Categoryfinal 
		, Month
		, IFRSGrossCarryingAmountPrvM
from `{env_prefix}-cdh-delivery`.repoint.core_modifications
where ModificationDate > (select max(RestructuringDate) from corestatichistoricalmodificationbeforemigration) 
AND ModificationDate <= {run_date!r}
'''
)
Core_Modifications.createOrReplaceTempView("Core_Modifications")

dexposure= spark.sql(
f'''
select
     ExposureKey
    ,CDI
    ,Bucket
from `{env_prefix}-cdh`.primary.exposure
where BusinessDate = {run_date!r}
and PortfolioName = 'Core Retail'
and PartyRelationShipType = 'Primary Owner'
and IsActiveFlag = true
'''
)
dexposure.createOrReplaceTempView("dexposure")

unionfile= spark.sql(
f'''
select 
     uf.ExposureNumber
    ,uf.BucketCategoryWithMateriality  
    ,uf.SnapshotDate
    ,uf.EBAStatus
from `{env_prefix}-cdh`.externalfiles.extunionfile uf
join (select SnapshotDate, max(IngestionTimestamp) IngestionTimestamp from `{env_prefix}-cdh`.externalfiles.extunionfile group by SnapshotDate) uf0
on uf.IngestionTimestamp = uf0.IngestionTimestamp and uf.SnapshotDate = uf0.SnapshotDate
where uf.SnapshotDate = (select max(SnapshotDate) from `{env_prefix}-cdh`.externalfiles.extunionfile where SnapshotDate < {run_date!r})
''')
unionfile.createOrReplaceTempView("unionfile")

##### AssignmentPool

In [0]:
%python
AssignmentPool = spark.sql(
f'''
Select * from
(SELECT 
      BusinessDate
      ,ExposureKey
      ,BalanceGroup
      ,SegmentDPO
      ,DenouncedFlag
      ,EAP
      ,Balance
      ,EBAStatus
      ,DPOModified
      ,ModificationMonth
      ,Segments
      ,ActiveCurrentMonthexclImplem
      ,CASE 
  WHEN CAST(SUBSTRING(Segments, 7, 2) AS STRING) NOT LIKE '%\_%' THEN 
    CONCAT(
      RIGHT(TRIM(SUBSTRING(Segments, 1, 16)), 4),
      LEFT(RIGHT(REPLACE(TRIM(SUBSTRING(Segments, 1, 16)), '.', ''), 6), 2)
    )
  ELSE 
    CONCAT(
      RIGHT(TRIM(SUBSTRING(Segments, 1, 15)), 4),
      LEFT(RIGHT(REPLACE(TRIM(SUBSTRING(Segments, 1, 15)), '.', ''), 6), 2)
    )
END AS SegmentMonth
      ,RANK() OVER (
  PARTITION BY ExposureKey 
  ORDER BY CONCAT(
    RIGHT(TRIM(SUBSTRING(Segments, 1, 15)), 4),
    LEFT(RIGHT(REPLACE(TRIM(SUBSTRING(Segments, 1, 15)), '.', ''), 6), 2)
  ) DESC
) AS Rank
FROM AssignmentPoolDPOCampaign) A
WHERE A.Rank = 1
'''
)
AssignmentPool.createOrReplaceTempView("AssignmentPool")

##### Activities

In [0]:
%python
Activities = spark.sql(
f'''
SELECT DISTINCT		
				  CA.ExposureKey
				,	CA.InteractionTypeKey
				,	AM.Category
				, AM.ActivityCode
				,	AM.Subcategory
				,	AM.Subcategory2
				,	CA.InteractionDate
				,	LEFT(AM.Rank,1) AS ActivityPriority
    			,	RANK() OVER(PARTITION BY CA.ExposureKey ORDER BY CA.InteractionDate DESC, LEFT(AM.Rank,1) DESC, CA.InteractionTypeKey DESC) Rank
		FROM interaction CA
		INNER JOIN `{env_prefix}-reportinputfiles`.repoint.core_campaignactivitymapping AM ON AM.Code = CA.InteractionTypeKey
''')
Activities.createOrReplaceTempView("Activities")

##### Modifications

In [0]:
%python
Modifications = spark.sql(
f'''
SELECT DISTINCT
				  a.*
		FROM (
    SELECT 
				  ModificationDate
				, OldExposureKey
				, NewExposureKey 
				, AssetSegment
				, ModificationAmount 
				, HaircutAmount 
				, Category 
				, Categoryfinal 
				, Month
				, IFRSGrossCarryingAmountPrvM
				, RANK() OVER(PARTITION BY ModificationDate,OldExposureKey,NewExposureKey  ORDER BY BusinessDate DESC) AS Ranking 
			FROM Core_Modifications m
   
   UNION ALL

   	SELECT 			  RestructuringDate	    AS ModificationDate
					, OldExposureKey 
					, NewExposureKey 
					, BusinessUnit					AS AssetSegment
					, RestructuringAmount   AS ModificationAmount 
					, HaircutAmount 
					, Category              
					, CategoryFinal
					, Month
					, IFRSGrossCarryingAmountPreviousMonthOldExposureKey AS IFRSGrossCarryingAmountPrvM
					, 1 AS Ranking
			FROM corestatichistoricalmodificationbeforemigration  hm
		)
		 AS a
    INNER JOIN AssignmentPool asp on asp.ExposureKey = a.OldExposureKey 
		WHERE a.Ranking =1
		AND (a.Category LIKE 'D1%' OR a.Category LIKE 'C6%')
		AND a.Month >= ASP.SegmentMonth
      '''
    )
Modifications.createOrReplaceTempView("Modifications")    

##### Applications

In [0]:
%python
Applications = spark.sql(
f'''
		SELECT npe.*
		FROM npeplanningdata npe
		INNER JOIN AssignmentPool asp on asp.ExposureKey = npe.ExposureKey
		WHERE (npe.ModificationType LIKE 'D1%' OR npe.ModificationType LIKE 'C6%')
  	AND NPE.DateEvaluation >= DATE_SUB(to_date(CONCAT(ASP.SegmentMonth, '01'), 'yyyyMMdd'), 5)
      '''
    )
Applications.createOrReplaceTempView("Applications")    

##### Temp_DPO

In [0]:
%python
Temp_DPO = spark.sql(
f'''
SELECT 
				ASP.ExposureKey
			,	MODS.NewExposureKey
			,	ASP.BalanceGroup
			,	ASP.SegmentDPO
			,	CASE
					WHEN ASP.EAP LIKE '%Positive%' THEN 'Positive'
					WHEN ASP.EAP LIKE '%Negative%' THEN 'Negative'
					WHEN ASP.SegmentDPO LIKE '%Dpo_PE%' THEN 'DPO PE'
				END													AS EAPClass
			,	CASE
					WHEN ASP.EAP LIKE '%Positive%' THEN 'Positive'
					WHEN ASP.DenouncedFlag = 'Non Denounced' THEN 'Non Denounced'
				END													AS AccountStatus
			,	IF(ASP.ActiveCurrentMonthexclImplem='',NULL,ASP.ActiveCurrentMonthexclImplem) AS Active
			,	ASP.Segments										AS Segment
			,	ASP.Balance AS Balance
			,	CASE WHEN TR.Category <> 'Rest Communications' THEN TR.InteractionTypeKey END AS InteractionTypeKey
			,	CASE WHEN TR.Category <> 'Rest Communications' THEN TR.ActivityCode END AS ActivityCode
			,	CASE WHEN TR.Category <> 'Rest Communications' THEN TR.InteractionDate END AS InteractionDate
			,	CASE WHEN TR.Category <> 'Rest Communications' THEN TR.Category END AS Category
			,	CASE WHEN TR.Category <> 'Rest Communications' THEN TR.Subcategory END AS Subcategory
      		,	CASE WHEN ASP.ExposureKey = NPE.ExposureKey THEN 'Positive' 
      			 WHEN TR.Category <> 'Rest Communications' THEN NULL
          	 	 ELSE TR.Category 
        		END AS NewCategory
    		,	CASE WHEN ASP.ExposureKey = NPE.ExposureKey  THEN 'Positive with application'
      			 	WHEN TR.Category <> 'Rest Communications' THEN NULL 
     				 ELSE TR.Subcategory 
        		END AS NewSubcategory
			,	CASE WHEN TR.Category = 'Rest Communications' THEN 'Yes' END AS RestComms
			,	CASE WHEN MODS.OldExposureKey IS NOT NULL THEN 'Y' END AS Modification
			,	CAST(MODS.ModificationDate AS DATE)					AS ModificationDate
			,	MODS.Category										AS ModsProduct
			,	MODS.HaircutAmount									AS ModsHaircut
			,	MODS.ModificationAmount								AS ModsAmount
			,	MODS.IFRSGrossCarryingAmountPrvM					AS ModsAmountBeforeModification
			,	NPE.Status
    		,	CASE WHEN LEFT(NPE.ModificationType,2) IS NULL AND TR.InteractionTypeKey in ('536','533','534','4103','4104') THEN 'C6'
          			 WHEN LEFT(NPE.ModificationType,2) IS NULL AND TR.InteractionTypeKey =  '578' THEN 'D1'
          			 ELSE LEFT(NPE.ModificationType,2) 
    			END AS DPOproduct
			,	NPE.ModificationType
			,	NPE.ApplicationNumber
			,	NPE.Decision
			,	NPE.ApplicationStepUpdated							AS CurrentStepUpdated
			,	CAST(NULL AS VARCHAR(255))							AS ApplicationStep
			,	NPE.DateEvaluation
			,	CASE 
					WHEN NPE.DateEvaluation IS NOT NULL THEN 'Yes'
				END													AS ApplicationsFlag
			,	CASE 
					WHEN MODS.OldExposureKey IS NOT NULL THEN 'Yes'
				END													AS ModificationFlag
			,	CASE WHEN MODS.OldExposureKey IS NOT NULL THEN de.Bucket END AS OperationalBucket
			--,	CASE WHEN MODS.OldExposureKey IS NOT NULL THEN u.BucketCategoryWithMateriality END AS RiskBucket
       		,	CASE WHEN MODS.OldExposureKey IS NOT NULL AND MODS.NewExposureKey = u1.ExposureNumber THEN u1.BucketCategoryWithMateriality 
          	 		 WHEN MODS.OldExposureKey IS NOT NULL AND ASP.ExposureKey = u.ExposureNumber THEN u.BucketCategoryWithMateriality
      			END AS RiskBucket
			,	IF(ASP.EBAStatus ='',NULL,ASP.EBAStatus)			AS EBAStatus
			,	CASE
					WHEN ASP.DenouncedFlag = 'Non Denounced' THEN 'Non Denounced'
					WHEN ASP.DenouncedFlag = 'Denounced' THEN 'Denounced'
					WHEN ASP.SegmentDPO ILIKE '%DPO%' AND ASP.BalanceGroup != '<3κ' THEN 'DPO PE'
				END													AS DenouncedStatus
			,	CASE WHEN TR.Category <> 'Rest Communications' THEN TR.Subcategory2 END AS Subcategory2
		FROM AssignmentPool								ASP
		LEFT JOIN Modifications							MODS ON MODS.OldExposureKey = ASP.ExposureKey
		LEFT JOIN Applications							NPE ON NPE.ExposureKey = ASP.ExposureKey
  	LEFT JOIN Activities							TR ON TR.ExposureKey = ASP.ExposureKey AND TR.Rank = 1
    LEFT JOIN dexposure 							DE on de.ExposureKey = MODS.NewExposureKey
		LEFT JOIN unionfile 							U on ASP.ExposureKey = U.ExposureNumber
		LEFT JOIN unionfile 							U1 on MODS.NewExposureKey= U1.ExposureNumber
      '''
    )
Temp_DPO.createOrReplaceTempView("Temp_DPO")   

##### DPOCampaign

In [0]:
%python
DPOCampaign = spark.sql(
f'''
SELECT 
			  {run_date!r} AS BusinessDate
      ,	ExposureKey
			,	ifnull(NewExposureKey,ExposureKey) AS NewExposureKey
			,	BalanceGroup
			,	SegmentDPO
			,	EAPClass
			,	AccountStatus
			,	Active
			,	Segment
			,	try_cast(trim(regexp_replace(replace(replace(Balance,'.',''),',','.'), '[^0-9.]', '')) as decimal(19,2)) as Balance
   			,	InteractionTypeKey
			,	ActivityCode
			,	InteractionDate
            ,   CASE WHEN Status = 'Ακυρωμένη' OR Category = 'negative' THEN 'Negative'
                     WHEN SubCategory = 'Without application' AND DPOproduct = 'D1' THEN 'Positive'
                     WHEN DPOproduct = 'D1' AND InteractionTypeKey IN ('4100', '516', '531') THEN 'Positive'
                     ELSE NewCategory
                END AS Category
			,	NewSubcategory AS Subcategory 
			,	RestComms
			,	Modification
			,	ModificationDate
			,	ModsProduct
			,	ModsHaircut
			,	ModsAmount
			,	ModsAmountBeforeModification
			,	Status
    		,	DPOproduct
			,	ModificationType
			,	ApplicationNumber
			,	Decision
			,	CurrentStepUpdated
            ,   CASE WHEN NewCategory ilike 'positive' then
                     CASE WHEN NewSubcategory NOT LIKE '%Negative%' AND Modification IS NOT NULL THEN '0a.Disburshed'
			              		  WHEN NewSubcategory NOT LIKE '%Negative%' AND Modification IS NULL AND Decision='2.ΑΚΥΡΩΣΗ' THEN '0b.Canceled'
			              		  WHEN NewSubcategory='Without application' AND Modification IS NULL AND (Decision IS NULL OR Decision<>'2.ΑΚΥΡΩΣΗ') THEN '0c.PosNoAppl'
			              		  WHEN NewSubcategory='Positive with application' AND Modification IS NULL AND (Decision IS NULL OR Decision<>'2.ΑΚΥΡΩΣΗ') THEN CurrentStepUpdated
			              		  ELSE ApplicationStep
                     END
                     ELSE ApplicationStep
                END AS ApplicationStep
			,	DateEvaluation
			,	ApplicationsFlag
			,	ModificationFlag
			,	OperationalBucket
			,	RiskBucket
			,	EBAStatus
			,	DenouncedStatus
			,	Subcategory2
		FROM Temp_DPO
      '''
    )
DPOCampaign.createOrReplaceTempView("DPOCampaign")   

##### final_df

In [0]:
%python
final_df = spark.sql(
f'''
SELECT 
    CAST(BusinessDate                        AS DATE)      AS BusinessDate,
    CAST(ExposureKey                         AS STRING)    AS ExposureKey,
    CAST(NewExposureKey                      AS STRING)    AS NewExposureKey,
    CAST(BalanceGroup                        AS STRING)    AS BalanceGroup,
    CAST(SegmentDPO                          AS STRING)    AS SegmentDPO,
    CAST(EAPClass                            AS STRING)    AS EAPClass,
    CAST(AccountStatus                       AS STRING)    AS AccountStatus,
    CAST(Active                              AS STRING)    AS Active,
    CAST(Segment                             AS STRING)    AS Segment,
    CAST(Balance                             AS DECIMAL(19,2)) AS Balance,
    CAST(InteractionTypeKey                  AS STRING)    AS InteractionTypeKey,
    CAST(ActivityCode                        AS STRING)    AS ActivityCode,
    CAST(InteractionDate                     AS DATE)      AS InteractionDate,
    CAST(Category                            AS STRING)    AS Category,
    CAST(Subcategory                         AS STRING)    AS Subcategory,
    CAST(RestComms                           AS STRING)    AS RestComms,
    CAST(Modification                        AS STRING)    AS Modification,
    CAST(ModificationDate                    AS DATE)      AS ModificationDate,
    CAST(ModsProduct                         AS STRING)    AS ModsProduct,
    CAST(ModsHaircut                         AS DECIMAL(19,2))    AS ModsHaircut,
    CAST(ModsAmount                          AS DECIMAL(19,2))    AS ModsAmount,
    CAST(ModsAmountBeforeModification        AS DECIMAL(19,2))    AS ModsAmountBeforeModification,
    CAST(Status                              AS STRING)    AS Status,
    CAST(DPOProduct                          AS STRING)    AS DPOProduct,
    CAST(ModificationType                    AS STRING)    AS ModificationType,
    CAST(ApplicationNumber                   AS STRING)    AS ApplicationNumber,
    CAST(Decision                            AS STRING)    AS Decision,
    CAST(CurrentStepUpdated                  AS STRING)    AS CurrentStepUpdated,
    CAST(ApplicationStep                     AS STRING)    AS ApplicationStep,
    CAST(DateEvaluation                      AS DATE)      AS DateEvaluation,
    CAST(ApplicationsFlag                    AS STRING)    AS ApplicationsFlag,
    CAST(ModificationFlag                    AS STRING)    AS ModificationFlag,
    CAST(OperationalBucket                   AS STRING)    AS OperationalBucket,
    CAST(RiskBucket                          AS STRING)    AS RiskBucket,
    CAST(EBAStatus                           AS STRING)    AS EBAStatus,
    CAST(DenouncedStatus                     AS STRING)    AS DenouncedStatus,
    CAST(Subcategory2                        AS STRING)    AS Subcategory2,
    CAST(getdate()                           AS TIMESTAMP) AS InsertTimeStamp
FROM DPOCampaign A
'''
)
final_df.createOrReplaceTempView("final_df")

In [0]:
%python
if final_df.rdd.isEmpty():
    raise ValueError(f"Core_DPOCampaign: final DataFrame is empty - aborting write to {delivery_table_path} to avoid overwriting with no data")
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(delivery_table_path) # in order to write in delta_lake

In [0]:
%python
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[col(c) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = '{run_date}'
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = ["ExposureKey", "NewExposureKey"]
check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="warning")

In [0]:
%python
import re
def rename_and_cleanup_files(output_path, file_name):
    part_file_regex = re.compile(r'part-.*')
    system_file_regex = re.compile(r'^_(committed_|started_|SUCCESS)')
 
    # Move the part file to the final filename.
    files = dbutils.fs.ls(output_path)
    for file in files:
        print(f"file: {file.name}")
        if part_file_regex.match(file.name):
            original_path = file.path
            new_path = output_path + file_name
            dbutils.fs.mv(original_path, new_path)
 
    # Remove system files from the common folder.
    files = dbutils.fs.ls(output_path)
    for file in files:
        if system_file_regex.match(file.name):
            dbutils.fs.rm(file.path, recurse=False)
 
# Example usage:
# rename_and_cleanup_files(output_path, file_name)

In [0]:
%python
DPOCampaign = spark.sql(
f'''
SELECT BusinessDate ReferenceDate
      , ExposureKey Account_No
			, SegmentDPO AS `Segment DPO`
			, Balance
			, BalanceGroup AS `Balance Group`
			, ModificationDate AS `Ημερομηνία Ρύθμισης`
FROM {delivery_table_path}
WHERE BusinessDate = {run_date!r}
  AND  ModificationFlag ='Yes'
GROUP BY  BusinessDate 
        , ExposureKey 
			  , SegmentDPO 
			  , Balance
			  , BalanceGroup 
			  , ModificationDate
'''
)
DPOCampaign.createOrReplaceTempView("DPOCampaign")

In [0]:
%python
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/Core/DPOCampaign/"
file_name = "DPOCampaign_" +run_date+ ".csv"
DPOCampaign.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}")
rename_and_cleanup_files(output_path, file_name)